# Analise e Respostas as Perguntas de Negócio
Nesse notebook iremos responder as perguntas iniciais que fizemos antes do inicio do projeto:

*OBS: Índice de pontualidade (I.P.): `Número de Voos pontuais na [etapa] / Número total de voos Realizados`*

1. **Qual o I.P. de partida e chegada, por companhia, em todo o período? Como cada uma se compara a média geral?** 
1. **Qual o I.P. de partida e chegada, por companhia, ano a ano (incluindo o ano incompleto de 2026)? Existe alguma piora ou melhora nesses índices?**
1. **Qual o I.P. de partida e chegada por Rota? Quais são as 10 melhores e 10 piores?**
1. **Qual o I.P de partida e chegada por Aeroporto? Quais são os 10 melhores e os 10 piores?**
1. **Como é o I.P de partida e chegada por mês? Existe algum ciclo ou sazonalidade?**
1. **Como é o I.P de partida e chegada por dia da semana? Existe alguma tendência?**
1. **Como é o I.P de partida e chegada por horário de partida do voo? Horários com mais partidas ou chegadas tem mais atrasos?**
1. **Qual é a média de tempo de atraso de partida e de chegada, por Companhia?**
1. **Qual é a média de tempo de atraso de partida e de chegada, por Rota?**
1. **Qual é a média de tempo de atraso de partida e de chegada, por Aeroporto?**

## 1. Qual o I.P. de partida e chegada, por companhia, em todo o período? Como cada uma se compara a média geral?

In [0]:
%sql
WITH pontualidade AS (
	SELECT
		de.empresa as empresa
		,COUNT(*) AS viagens
		,SUM(IF(fv.pontual_partida, 1, 0)) AS partidas_pontuais
		,SUM(IF(fv.pontual_chegada, 1, 0)) AS chegadas_pontuais
		, ROUND(TRY_DIVIDE(SUM(IF(fv.pontual_partida, 1, 0)), COUNT(*)), 3) AS IP_partida
		,ROUND(TRY_DIVIDE(SUM(IF(fv.pontual_chegada, 1, 0)), COUNT(*)), 3)  AS IP_chegada
	FROM
		mvp_engenharia_dados.gold.fato_voos fv
			JOIN mvp_engenharia_dados.gold.dim_empresas de
				ON de.sk_empresa = fv.sk_empresa

	GROUP BY
		de.empresa
)
SELECT
	*
    , ROUND(TRY_DIVIDE(SUM(partidas_pontuais) OVER (), SUM(viagens) OVER ()), 3)  AS IP_partida_geral
    , ROUND(TRY_DIVIDE(SUM(chegadas_pontuais) OVER (), SUM(viagens) OVER ()), 3) AS IP_chegada_geral
    , ROUND(IP_partida - TRY_DIVIDE(SUM(partidas_pontuais) OVER (), SUM(viagens) OVER ()), 3) as delta_IP_partida
    , ROUND(IP_chegada - TRY_DIVIDE(SUM(chegadas_pontuais) OVER (), SUM(viagens) OVER ()), 3) as delta_IP_chegada
FROM
	pontualidade
ORDER BY IP_partida DESC, IP_chegada DESC


### Ranking de Indice de Pontualidade em todo o período (Jan 2023 a Jul 2026)

|empresa|viagens|partidas_pontuais|chegadas_pontuais|IP_partida|IP_chegada|IP_partida_geral|IP_chegada_geral|delta_IP_partida|delta_IP_chegada|
|---|---|---|---|---|---|---|---|---|---|
|TAM (TAM)|888,585|823,053|823,185|92.6%|92.6%|92.2%|92.1%|0.4%|0.5%|
|GOL (GLO)|751,816|696,367|694,858|92.6%|92.4%|92.2%|92.1%|0.4%|0.3%|
|AZUL (AZU)|993,440|916,693|915,772|92.3%|92.2%|92.2%|92.1%|0.1%|0.1%|
|SIDERAL (SID)|1,175|1,037|998|88.3%|84.9%|92.2%|92.1%|-3.9%|-7.2%|
|AZUL (ACN)|54,301|47,791|47,421|88%|87.3%|92.2%|92.1%|-4.2%|-4.8%|
|ATA (ABJ)|3,387|2,904|2,889|85.7%|85.3%|92.2%|92.1%|-6.5%|-6.8%|
|PLACAR (PLS)|62|52|46|83.9%|74.2%|92.2%|92.1%|-8.3%|-17.9%|
|VOEPASS (PTB)|32,318|26,437|25,951|81.8%|80.3%|92.2%|92.1%|-10.4%|-11.8%|
|OMNI (OMI)|178|136|130|76.4%|73%|92.2%|92.1%|-15.8%|-19.1%|
|MAP (PAM)|6,459|4,934|4,711|76.4%|72.9%|92.2%|92.1%|-15.8%|-19.2%|
|APUÍ (CQB)|599|456|422|76.1%|70.5%|92.2%|92.1%|-16.1%|-21.6%|
|AVION (AEB)|11|7|6|63.6%|54.5%|92.2%|92.1%|-28.6%|-37.6%|

> TAM (Ou Latam como é seu nome Comercial) tem o melhor Índice de Pontualidade tanto de partidas com `92.6%` quanto de chegadas com também `92.6%`.

> A média geral de partidas é de `92.2%`, enquanto que a de chegadas é de `92.1%`.


## 2. Qual o I.P. de partida e chegada, por companhia, ano a ano (incluindo o ano incompleto de 2026)? Existe alguma piora ou melhora nesses índices?

In [0]:
%sql
-- pontualidade de partida
WITH pontualidade AS (
	SELECT
		de.empresa
		,dc.ano
		,ROUND(TRY_DIVIDE(SUM(IF(fv.pontual_partida, 1, 0)), COUNT(*)), 3) AS IP_partida

	FROM
		mvp_engenharia_dados.gold.fato_voos fv
			JOIN mvp_engenharia_dados.gold.dim_empresas de
				ON de.sk_empresa = fv.sk_empresa
			JOIN mvp_engenharia_dados.gold.dim_calendario dc
				ON dc.sk_data = fv.sk_data_partida

	GROUP BY
		de.empresa
		,dc.ano
)
SELECT
	*

FROM
	pontualidade PIVOT (FIRST(IP_partida) FOR ano IN (2023, 2024, 2025, 2026))
ORDER BY
	`2026` DESC

### Pontualidade por Empresas Ano a Ano (Partida)

|empresa|2023|2024|2025|2026|
|---|---|---|---|---|
|ATA (ABJ)|73.2%|69.8%|98.3%|98.6%|
|APUÍ (CQB)|null|77.5%|63.9%|94.8%|
|GOL (GLO)|90.1%|92.7%|94.1%|94%|
|AZUL (AZU)|93.5%|91.3%|92.2%|91.9%|
|TAM (TAM)|92.7%|93.5%|92.3%|91.8%|
|AZUL (ACN)|88.1%|87.3%|88%|90.1%|
|SIDERAL (SID)|88.1%|88.3%|89.9%|85%|
|PLACAR (PLS)|null|null|87.5%|81.6%|
|AVION (AEB)|null|null|71.4%|50%|
|OMNI (OMI)|76.4%|null|null|null|
|VOEPASS (PTB)|83.7%|80.4%|77.1%|null|
|MAP (PAM)|72.5%|78.8%|85.2%|null|

> É possível notar que algumas empresas tiveram melhora expressiva ao longo dos anos, **ATA** por exemplo estava com apenas `73.2%` em 2023 e agora em 2026 encontra-se com `98.6%` de pontualidade de partida.

> Outras como APUI , AZUL (ACN) também tiveram melhora notavel.

> Algumas se manteram estaveis com poucas ocilações, mas pouquissimas tiveram uma piora muito expressiva.

In [0]:
%sql
-- pontualidade de chegada
WITH pontualidade AS (
	SELECT
		de.empresa
		,dc.ano
		,ROUND(TRY_DIVIDE(SUM(IF(fv.pontual_chegada, 1, 0)), COUNT(*)), 3) AS IP_chegada

	FROM
		mvp_engenharia_dados.gold.fato_voos fv
			JOIN mvp_engenharia_dados.gold.dim_empresas de
				ON de.sk_empresa = fv.sk_empresa
			JOIN mvp_engenharia_dados.gold.dim_calendario dc
				ON dc.sk_data = fv.sk_data_chegada

	GROUP BY
		de.empresa
		,dc.ano
)
SELECT
	*

FROM
	pontualidade PIVOT (FIRST(IP_chegada) FOR ano IN (2023, 2024, 2025, 2026))
ORDER BY
	`2026` DESC

### Pontualidade por Empresas Ano a Ano (Chegada)

|empresa|2023|2024|2025|2026|
|---|---|---|---|---|
|ATA (ABJ)|73.2%|69.8%|98.1%|96.7%|
|GOL (GLO)|89%|92.5%|94.5%|94.1%|
|AZUL (AZU)|93.3%|91.1%|92.2%|91.9%|
|APUÍ (CQB)|null|67.5%|60%|91.5%|
|TAM (TAM)|92.5%|93.7%|92.5%|91.5%|
|AZUL (ACN)|86.8%|86.4%|87.9%|90.5%|
|SIDERAL (SID)|85.8%|82.7%|87%|82.8%|
|PLACAR (PLS)|null|null|87.5%|65.8%|
|AVION (AEB)|null|null|71.4%|25%|
|OMNI (OMI)|73%|null|null|null|
|VOEPASS (PTB)|82.2%|78.8%|76.2%|null|
|MAP (PAM)|68.1%|76.1%|83%|null|

> Semelhante a evolução na pontualidade de partida, a de chegada mantem posições parecidas no geral, ou seja a tendência da pontualidade de partida é acopanhanda pela de chegada.

## 3. Qual o I.P. de partida e chegada por Rota? Quais são as 10 melhores e 10 piores?

#### Antes de executar essa analise vamos definir um piso minimo de quantidade de viagens.
Usaremos um intervalo de confiança com os seguintes parametros:

`z = 1.96` - 95% de confiança.

`p = 0.922` - 92.2% é a taxa média geral de pontualidade na base.

 `e = 0.03` - A margem de erro fica em 3 pontos percentuais.

In [0]:
import math

z = 1.96       # 95% de confiança
p = 0.922      # taxa esperada de pontualidade
e = 0.03       # margem de erro de 3 p.p.

n = (z**2 * p * (1 - p)) / (e**2)

print(f"n calculado: {n:.1f}")
print(f"n mínimo: {math.ceil(n)}")

> Com o resultado do cálculo obtemos um N mínimo de **307 amostras**.

In [0]:
%sql
WITH rotas_validas AS (
	SELECT
		fv.sk_aeroporto_origem
		,fv.sk_aeroporto_destino
		,COUNT(*) AS viagens

	FROM
		mvp_engenharia_dados.gold.fato_voos fv
			JOIN mvp_engenharia_dados.gold.dim_calendario d
				ON d.sk_data = fv.sk_data_partida

	GROUP BY
		fv.sk_aeroporto_origem
		,fv.sk_aeroporto_destino

	HAVING
		viagens >= 307
)
SELECT
	CONCAT(ori.aerodromo, ' X ', des.aerodromo) AS rota
	,COUNT(*) AS viagens
	,SUM(IF(fv.pontual_partida, 1, 0)) AS partidas_pontuais
	,ROUND(TRY_DIVIDE(SUM(IF(fv.pontual_partida, 1, 0)), COUNT(*)), 3) AS IP_partida

FROM
	rotas_validas rv
		JOIN mvp_engenharia_dados.gold.fato_voos fv
			ON fv.sk_aeroporto_origem = rv.sk_aeroporto_origem
			AND fv.sk_aeroporto_destino = rv.sk_aeroporto_destino
		JOIN mvp_engenharia_dados.gold.dim_aerodromos ori
			ON ori.sk_aerodromo = fv.sk_aeroporto_origem
		JOIN mvp_engenharia_dados.gold.dim_aerodromos des
			ON des.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	ori.aerodromo
	,des.aerodromo
ORDER BY
	IP_partida DESC
	,viagens DESC
LIMIT 10

### 10 Rotas com **Melhor** Pontualidade de partida.
_Ao menos 307 amostras._

|rota|viagens|partidas_pontuais|IP_partida|
|---|---|---|---|
|CANOAS (SBCO) X VIRACOPOS (SBKP)|431|429|99.5%|
|JOÃO CORRÊA DA ROCHA (SBMA) X PRESIDENTE JUSCELINO KUBITSCHEK (SBBR)|2,367|2,343|99%|
|CANOAS (SBCO) X GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO (SBGR)|662|655|98.9%|
|BRIGADEIRO LYSIAS RODRIGUES (SBPJ) X SANTA GENOVEVA (SBGO)|1,256|1,231|98%|
|VIRACOPOS (SBKP) X CANOAS (SBCO)|430|421|97.9%|
|PROFESSOR URBANO ERNESTO STUMPF (SBSJ) X RIO DE JANEIRO/GALEÃO - ANTÔNIO CARLOS JOBIM (SBGL)|383|374|97.7%|
|PRESIDENTE JUSCELINO KUBITSCHEK (SBBR) X CATARATAS (SBFI)|346|338|97.7%|
|PRESIDENTE JUSCELINO KUBITSCHEK (SBBR) X PREFEITO RENATO MOREIRA (SBIZ)|1,213|1,183|97.5%|
|EURICO DE AGUIAR SALLES (SBVT) X AEROPORTO INTERNACIONAL RECIFE/GUARARAPES - GILBERTO FREYRE (SBRF)|1,131|1,103|97.5%|
|NELSON RODRIGUES GUIMARÃES (SBCN) X SÃO PAULO/CONGONHAS - DEPUTADO FREITAS NOBRE (SBSP)|558|544|97.5%|


In [0]:
%sql
WITH rotas_validas AS (
	SELECT
		fv.sk_aeroporto_origem
		,fv.sk_aeroporto_destino
		,COUNT(*) AS viagens

	FROM
		mvp_engenharia_dados.gold.fato_voos fv
			JOIN mvp_engenharia_dados.gold.dim_calendario d
				ON d.sk_data = fv.sk_data_partida

	GROUP BY
		fv.sk_aeroporto_origem
		,fv.sk_aeroporto_destino

	HAVING
		viagens >= 307
)
SELECT
	CONCAT(ori.aerodromo, ' X ', des.aerodromo) AS rota
	,COUNT(*) AS viagens
	,SUM(IF(fv.pontual_partida, 1, 0)) AS partidas_pontuais
	,ROUND(TRY_DIVIDE(SUM(IF(fv.pontual_partida, 1, 0)), COUNT(*)), 3) AS IP_partida

FROM
	rotas_validas rv
		JOIN mvp_engenharia_dados.gold.fato_voos fv
			ON fv.sk_aeroporto_origem = rv.sk_aeroporto_origem
			AND fv.sk_aeroporto_destino = rv.sk_aeroporto_destino
		JOIN mvp_engenharia_dados.gold.dim_aerodromos ori
			ON ori.sk_aerodromo = fv.sk_aeroporto_origem
		JOIN mvp_engenharia_dados.gold.dim_aerodromos des
			ON des.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	ori.aerodromo
	,des.aerodromo
ORDER BY
	IP_partida ASC
	,viagens DESC
LIMIT 10

### 10 Rotas com **Pior** Pontualidade de partida.
_Ao menos 307 amostras._

|rota|viagens|partidas_pontuais|IP_partida|
|---|---|---|---|
|LEITE LOPES (SBRP) X PRESIDENTE JUSCELINO KUBITSCHEK (SBBR)|345|232|67.2%|
|SÃO PAULO/CONGONHAS - DEPUTADO FREITAS NOBRE (SBSP) X USIMINAS (SBIP)|767|560|73%|
|URUCU (SBUY) X EDUARDO GOMES (SBEG)|1743|1300|74.6%|
|LEITE LOPES (SBRP) X SANTOS DUMONT (SBRJ)|606|472|77.9%|
|LAURO CARNEIRO DE LOYOLA (SBJV) X SALGADO FILHO (SBPA)|335|261|77.9%|
|SANTOS DUMONT (SBRJ) X BARTOLOMEU LISANDRO (SBCP)|378|298|78.8%|
|SANTOS DUMONT (SBRJ) X LEITE LOPES (SBRP)|604|479|79.3%|
|MUNICIPAL DE LINHARES (SNLN) X TANCREDO NEVES (SBCF)|470|373|79.4%|
|EDUARDO GOMES (SBEG) X URUCU (SBUY)|1746|1391|79.7%|
|SÃO PAULO/CONGONHAS - DEPUTADO FREITAS NOBRE (SBSP) X REGIONAL SUL (SBJA)|465|373|80.2%|


In [0]:
%sql
WITH rotas_validas AS (
	SELECT
		fv.sk_aeroporto_origem
		,fv.sk_aeroporto_destino
		,COUNT(*) AS viagens

	FROM
		mvp_engenharia_dados.gold.fato_voos fv
			JOIN mvp_engenharia_dados.gold.dim_calendario d
				ON d.sk_data = fv.sk_data_partida

	GROUP BY
		fv.sk_aeroporto_origem
		,fv.sk_aeroporto_destino

	HAVING
		viagens >= 307
)
SELECT
	CONCAT(ori.aerodromo, ' X ', des.aerodromo) AS rota
	,COUNT(*) AS viagens
	,SUM(IF(fv.pontual_chegada, 1, 0)) AS chegadas_pontuais
	,ROUND(TRY_DIVIDE(SUM(IF(fv.pontual_chegada, 1, 0)), COUNT(*)), 3) AS IP_chegada

FROM
	rotas_validas rv
		JOIN mvp_engenharia_dados.gold.fato_voos fv
			ON fv.sk_aeroporto_origem = rv.sk_aeroporto_origem
			AND fv.sk_aeroporto_destino = rv.sk_aeroporto_destino
		JOIN mvp_engenharia_dados.gold.dim_aerodromos ori
			ON ori.sk_aerodromo = fv.sk_aeroporto_origem
		JOIN mvp_engenharia_dados.gold.dim_aerodromos des
			ON des.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	ori.aerodromo
	,des.aerodromo
ORDER BY
	IP_chegada DESC
	,viagens DESC
LIMIT 10

### 10 Rotas com **Melhor** Pontualidade de Chegada.
_Ao menos 307 viagens._

|rota|viagens|chegadas_pontuais|IP_chegada|
|---|---|---|---|
|CANOAS (SBCO) X VIRACOPOS (SBKP)|431|429|99.5%|
|JOÃO CORRÊA DA ROCHA (SBMA) X PRESIDENTE JUSCELINO KUBITSCHEK (SBBR)|2367|2341|98.9%|
|CANOAS (SBCO) X GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO (SBGR)|662|653|98.6%|
|PRESIDENTE JUSCELINO KUBITSCHEK (SBBR) X CATARATAS (SBFI)|346|340|98.3%|
|BRIGADEIRO LYSIAS RODRIGUES (SBPJ) X SANTA GENOVEVA (SBGO)|1256|1231|98%|
|EURICO DE AGUIAR SALLES (SBVT) X AEROPORTO INTERNACIONAL RECIFE/GUARARAPES - GILBERTO FREYRE (SBRF)|1131|1106|97.8%|
|VIRACOPOS (SBKP) X CANOAS (SBCO)|430|420|97.7%|
|PROFESSOR URBANO ERNESTO STUMPF (SBSJ) X RIO DE JANEIRO/GALEÃO - ANTÔNIO CARLOS JOBIM (SBGL)|383|374|97.7%|
|DEPUTADO LUÍS EDUARDO MAGALHÃES (SBSV) X ZUMBI DOS PALMARES (SBMO)|1324|1291|97.5%|
|PLÁCIDO DE CASTRO (SBRB) X PRESIDENTE JUSCELINO KUBITSCHEK (SBBR)|2979|2903|97.4%|


In [0]:
%sql
WITH rotas_validas AS (
	SELECT
		fv.sk_aeroporto_origem
		,fv.sk_aeroporto_destino
		,COUNT(*) AS viagens

	FROM
		mvp_engenharia_dados.gold.fato_voos fv
			JOIN mvp_engenharia_dados.gold.dim_calendario d
				ON d.sk_data = fv.sk_data_partida

	GROUP BY
		fv.sk_aeroporto_origem
		,fv.sk_aeroporto_destino

	HAVING
		viagens >= 307
)
SELECT
	CONCAT(ori.aerodromo, ' X ', des.aerodromo) AS rota
	,COUNT(*) AS viagens
	,SUM(IF(fv.pontual_chegada, 1, 0)) AS chegadas_pontuais
	,ROUND(TRY_DIVIDE(SUM(IF(fv.pontual_chegada, 1, 0)), COUNT(*)), 3) AS IP_chegada

FROM
	rotas_validas rv
		JOIN mvp_engenharia_dados.gold.fato_voos fv
			ON fv.sk_aeroporto_origem = rv.sk_aeroporto_origem
			AND fv.sk_aeroporto_destino = rv.sk_aeroporto_destino
		JOIN mvp_engenharia_dados.gold.dim_aerodromos ori
			ON ori.sk_aerodromo = fv.sk_aeroporto_origem
		JOIN mvp_engenharia_dados.gold.dim_aerodromos des
			ON des.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	ori.aerodromo
	,des.aerodromo
ORDER BY
	IP_chegada ASC
	,viagens DESC
LIMIT 10

### 10 Rotas com **Pior** Pontualidade de Chegada.
_Ao menos 307 viagens._

|rota|viagens|chegadas_pontuais|IP_chegada|
|---|---|---|---|
|LEITE LOPES (SBRP) X PRESIDENTE JUSCELINO KUBITSCHEK (SBBR)|345|224|64.9%|
|URUCU (SBUY) X EDUARDO GOMES (SBEG)|1743|1213|69.6%|
|SÃO PAULO/CONGONHAS - DEPUTADO FREITAS NOBRE (SBSP) X USIMINAS (SBIP)|767|548|71.4%|
|SÃO PAULO/CONGONHAS - DEPUTADO FREITAS NOBRE (SBSP) X REGIONAL SUL (SBJA)|465|349|75.1%|
|EDUARDO GOMES (SBEG) X URUCU (SBUY)|1746|1327|76%|
|LEITE LOPES (SBRP) X SANTOS DUMONT (SBRJ)|606|463|76.4%|
|SÃO PAULO/CONGONHAS - DEPUTADO FREITAS NOBRE (SBSP) X PRESIDENTE PRUDENTE (SBDN)|2171|1685|77.6%|
|LAURO CARNEIRO DE LOYOLA (SBJV) X SALGADO FILHO (SBPA)|335|260|77.6%|
|MUNICIPAL DE LINHARES (SNLN) X TANCREDO NEVES (SBCF)|470|369|78.5%|
|TANCREDO NEVES (SBCF) X MUNICIPAL DE LINHARES (SNLN)|474|377|79.5%|

## 4. Qual o I.P de partida e chegada por Aeroporto? Quais são os 10 melhores e os 10 piores?

#### Usaremos o mesmo intervalo de confiança definido anteriormente: 307 amostras minimas.

In [0]:
%sql
SELECT
	da.aerodromo
	,da.municipio
	,da.uf
	,COUNT(*) AS viagens
	,SUM(IF(pontual_partida, 1, 0)) AS partidas_pontuais
	,ROUND(TRY_DIVIDE(SUM(IF(pontual_partida, 1, 0)), COUNT(*)), 3) AS IP_partida

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos da
			ON da.sk_aerodromo = fv.sk_aeroporto_origem

GROUP BY
	da.aerodromo
	,da.municipio
	,da.uf

HAVING
	COUNT(*) >= 307
ORDER BY
	IP_partida DESC
	,viagens DESC
LIMIT 10

### Pontualidade de Partida por Aeroporto (10 Melhores)

|aerodromo|municipio|uf|viagens|partidas_pontuais|IP_partida|
|---|---|---|---|---|---|
|CANOAS (SBCO)|CANOAS|RIO GRANDE DO SUL|1,285|1,272|99%|
|PROFESSOR URBANO ERNESTO STUMPF (SBSJ)|SÃO JOSÉ DOS CAMPOS|SÃO PAULO|397|388|97.7%|
|JOÃO CORRÊA DA ROCHA (SBMA)|MARABÁ|PARÁ|5,642|5,451|96.6%|
|NELSON RODRIGUES GUIMARÃES (SBCN)|CALDAS NOVAS|GOIÁS|912|878|96.3%|
|BRIGADEIRO LYSIAS RODRIGUES (SBPJ)|PALMAS|TOCANTINS|9,726|9,352|96.2%|
|AERÓDROMO PÚBLICO DE BONITO (SBDB)|BONITO|MATO GROSSO DO SUL|1,224|1,174|95.9%|
|SENADOR PETRÔNIO PORTELLA (SBTE)|TERESINA|PIAUÍ|14,338|13,689|95.5%|
|AEROPORTO INTERNACIONAL SENADOR NILO COELHO (SBPL)|PETROLINA|PERNAMBUCO|7,292|6,965|95.5%|
|PLÁCIDO DE CASTRO (SBRB)|RIO BRANCO|ACRE|5,278|5,033|95.4%|
|PREFEITO RENATO MOREIRA (SBIZ)|IMPERATRIZ|MARANHÃO|5,054|4,816|95.3%|

In [0]:
%sql
SELECT
	da.aerodromo
	,da.municipio
	,da.uf
	,COUNT(*) AS viagens
	,SUM(IF(pontual_partida, 1, 0)) AS partidas_pontuais
	,ROUND(TRY_DIVIDE(SUM(IF(pontual_partida, 1, 0)), COUNT(*)), 3) AS IP_partida

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos da
			ON da.sk_aerodromo = fv.sk_aeroporto_origem

GROUP BY
	da.aerodromo
	,da.municipio
	,da.uf

HAVING
	COUNT(*) >= 307
ORDER BY
	IP_partida ASC
	,viagens DESC
LIMIT 10


### Pontualidade de Partida por Aeroporto (10 Piores)

|aerodromo|municipio|uf|viagens|partidas_pontuais|IP_partida|
|---|---|---|---|---|---|
|CARAUARI (SWCA)|CARAUARI|AMAZONAS|399|269|67.4%|
|URUCU (SBUY)|COARI|AMAZONAS|2,035|1,486|73%|
|MUNICIPAL DE LINHARES (SNLN)|LINHARES|ESPÍRITO SANTO|470|373|79.4%|
|OIAPOQUE (SBOI)|OIAPOQUE|AMAPÁ|433|352|81.3%|
|ARIPUANÃ (SSOU)|ARIPUANÃ|MATO GROSSO|684|569|83.2%|
|BARRA DO GARÇAS (SBBW)|BARRA DO GARÇAS|MATO GROSSO|653|544|83.3%|
|GUAÍRA (SSGY)|GUAÍRA|PARANÁ|392|327|83.4%|
|SALINAS (SNSS)|SALINAS|MINAS GERAIS|349|291|83.4%|
|LORENZO (SNCL)|CAIRU|BAHIA|1,434|1,204|84%|
|BARTOLOMEU DE GUSMÃO (SBAQ)|ARARAQUARA|SÃO PAULO|647|548|84.7%|

In [0]:
%sql
SELECT
	da.aerodromo
	, da.municipio
	, da.uf
	,COUNT(*) AS viagens
	,SUM(IF(pontual_chegada, 1, 0)) AS chegadas_pontuais
	,ROUND(TRY_DIVIDE(SUM(IF(pontual_chegada, 1, 0)), COUNT(*)), 3) AS IP_chegada

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos da
			ON da.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	da.aerodromo, da.municipio, da.uf

HAVING
	COUNT(*) >= 307
ORDER BY
	IP_chegada DESC
	,viagens DESC
LIMIT 10

### Pontualidade de Chegada por Aeroporto (10 Melhores)

|aerodromo|municipio|uf|viagens|chegadas_pontuais|IP_chegada|
|---|---|---|---|---|---|
|CANOAS (SBCO)|CANOAS|RIO GRANDE DO SUL|1,291|1,249|96.7%|
|CORUMBÁ (SBCR)|CORUMBÁ|MATO GROSSO DO SUL|622|595|95.7%|
|PROFESSOR URBANO ERNESTO STUMPF (SBSJ)|SÃO JOSÉ DOS CAMPOS|SÃO PAULO|383|366|95.6%|
|PAULO AFONSO (SBUF)|PAULO AFONSO|BAHIA|321|307|95.6%|
|ALTAMIRA (SBHT)|ALTAMIRA|PARÁ|1,502|1,429|95.1%|
|TRÊS LAGOAS (SBTG)|TRÊS LAGOAS|MATO GROSSO DO SUL|551|524|95.1%|
|MAESTRO WILSON FONSECA (SBSN)|SANTARÉM|PARÁ|7,441|7,072|95%|
|AERÓDROMO DE GUANAMBI (SNGI)|GUANAMBI|BAHIA|1,419|1,348|95%|
|CRUZEIRO DO SUL (SBCZ)|CRUZEIRO DO SUL|ACRE|704|669|95%|
|JOÃO CORRÊA DA ROCHA (SBMA)|MARABÁ|PARÁ|5,562|5,280|94.9%|

In [0]:
%sql
SELECT
	da.aerodromo
	, da.municipio
	, da.uf
	,COUNT(*) AS viagens
	,SUM(IF(pontual_chegada, 1, 0)) AS chegadas_pontuais
	,ROUND(TRY_DIVIDE(SUM(IF(pontual_chegada, 1, 0)), COUNT(*)), 3) AS IP_chegada

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos da
			ON da.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	da.aerodromo, da.municipio, da.uf

HAVING
	COUNT(*) >= 307 
ORDER BY
	IP_chegada ASC
	,viagens DESC
LIMIT 10

### Pontualidade de Chegada por Aeroporto (10 Piores)

|aerodromo|municipio|uf|viagens|chegadas_pontuais|IP_chegada|
|---|---|---|---|---|---|
|CARAUARI (SWCA)|CARAUARI|AMAZONAS|400|247|61.8%|
|URUCU (SBUY)|COARI|AMAZONAS|2,040|1,530|75%|
|MUNICIPAL DE LINHARES (SNLN)|LINHARES|ESPÍRITO SANTO|474|377|79.5%|
|ARIPUANÃ (SSOU)|ARIPUANÃ|MATO GROSSO|714|588|82.4%|
|SALINAS (SNSS)|SALINAS|MINAS GERAIS|353|292|82.7%|
|COARI (SWKO)|COARI|AMAZONAS|793|662|83.5%|
|RUBEM BERTA (SBUG)|URUGUAIANA|RIO GRANDE DO SUL|946|792|83.7%|
|MANICORÉ (SBMY)|MANICORÉ|AMAZONAS|589|494|83.9%|
|BARRA DO GARÇAS (SBBW)|BARRA DO GARÇAS|MATO GROSSO|660|556|84.2%|
|OIAPOQUE (SBOI)|OIAPOQUE|AMAPÁ|447|378|84.6%|

## 5. Como é o I.P de partida e chegada por mês? Existe algum ciclo ou sazonalidade?

In [0]:
%sql
WITH voos AS (
	SELECT
		d.ano
		,d.mes
        , d.nome_mes
		,COUNT(*) AS viagens
        , SUM(IF(pontual_partida, 1, 0)) AS partidas_pontuais
        , SUM(IF(pontual_chegada, 1, 0)) AS chegadas_pontuais
        , ROUND(TRY_DIVIDE(SUM(IF(pontual_partida, 1, 0)), COUNT(*)), 3) AS IP_partida
        , ROUND(TRY_DIVIDE(SUM(IF(pontual_chegada, 1, 0)), COUNT(*)), 3) AS IP_chegada

	FROM
		mvp_engenharia_dados.gold.fato_voos fv
			JOIN mvp_engenharia_dados.gold.dim_calendario d
				ON d.sk_data = fv.sk_data_partida

	GROUP BY
		d.ano
		,d.mes
		,d.nome_mes
)
SELECT
	nome_mes as mes_partida
    , avg(viagens) AS media_viagens
    , avg(partidas_pontuais) AS media_partidas_pontuais
    , avg(chegadas_pontuais) AS media_chegadas_pontuais
    , avg(IP_partida) AS media_IP_partida
    , avg(IP_chegada) AS media_IP_chegada
FROM
	voos
GROUP by mes, nome_mes
ORDER BY mes

### I.P. por Mês
_Tendência e Sazonalidae_

|mes_partida|media_viagens|media_partidas_pontuais|media_chegadas_pontuais|media_IP_partida|media_IP_chegada|
|---|---|---|---|---|---|
|Janeiro|66,983.25|61,338.5|61,151|91.55%|91.28%|
|Fevereiro|58,071.5|53,168.75|52,987.75|91.55%|91.23%|
|Março|63,756.25|59,517.5|59,437.75|93.35%|93.23%|
|Abril|61,898|57,504.5|57,253.75|92.9%|92.5%|
|Maio|63,431.75|59,928.25|59,884.5|94.47%|94.4%|
|Junho|61,944.75|57,763.5|57,624.75|93.28%|93.03%|
|Julho|67,593.25|63,270.25|63,230.5|93.6%|93.58%|
|Agosto|49,257|45,848.25|45,787.75|82.33%|82.22%|
|Setembro|63,093|58,794.67|58,726.33|93.17%|93.07%|
|Outubro|64,374.33|58,319|58,245.33|90.5%|90.4%|
|Novembro|60,968.67|54,756|54,883.33|89.8%|89.97%|
|Dezembro|65,093.33|56,966.67|57,131|87.57%|87.8%|

> Existe uma certa estabilidade ao longo do ano, com uma queda em abrupta em Agosto, o número se estabiliza novamente para depois voltar a cair em novembro e dezembro.  
> Tanto I.P. Partida quanto de Chegada seguem a mesma tendência.  
> O Melhor mês é o de Maio, e o pior é o de Agosto.

## 6. Como é o I.P de partida e chegada por dia da semana? Existe alguma tendência?

In [0]:
%sql
WITH voos AS (
	SELECT
		d.data
		,d.dia_semana
        , d.nome_dia_semana
		,COUNT(*) AS viagens
        , SUM(IF(pontual_partida, 1, 0)) AS partidas_pontuais
        , SUM(IF(pontual_chegada, 1, 0)) AS chegadas_pontuais
        , ROUND(TRY_DIVIDE(SUM(IF(pontual_partida, 1, 0)), COUNT(*)), 3) AS IP_partida
        , ROUND(TRY_DIVIDE(SUM(IF(pontual_chegada, 1, 0)), COUNT(*)), 3) AS IP_chegada

	FROM
		mvp_engenharia_dados.gold.fato_voos fv
			JOIN mvp_engenharia_dados.gold.dim_calendario d
				ON d.sk_data = fv.sk_data_partida

	GROUP BY
		d.data
		,d.dia_semana
        , d.nome_dia_semana
)
SELECT
	nome_dia_semana as dia_da_semana
    , avg(viagens) AS media_viagens
    , avg(partidas_pontuais) AS media_partidas_pontuais
    , avg(chegadas_pontuais) AS media_chegadas_pontuais
    , avg(IP_partida) AS media_IP_partida
    , avg(IP_chegada) AS media_IP_chegada
FROM
	voos
GROUP by dia_semana, nome_dia_semana
ORDER BY dia_semana

### Pontualidade de Partida e Chegada por dia da Semana da Partida

|dia_da_semana|media_viagens|media_partidas_pontuais|media_chegadas_pontuais|media_IP_partida|media_IP_chegada|
|---|---|---|---|---|---|
|Domingo|1,992.83|1,862.89|1,862.18|93.51%|93.47%|
|Segunda-feira|2,188.99|2,025.72|2,023.92|92.55%|92.47%|
|Terça-feira|2,147.71|1,990.2|1,985.25|92.67%|92.43%|
|Quarta-feira|2,147.2|1,976.95|1,970.07|92.02%|91.7%|
|Quinta-feira|2,147.84|1,956.91|1,950.36|91.08%|90.77%|
|Sexta-feira|2,162.64|1,960.27|1,958.36|90.7%|90.61%|
|Sábado|1,824.18|1,702.28|1,706.48|93.14%|93.37%|

> Há uma queda na pontualidade especificamente entre Quinta-feira e Sexta-feira, oque parece indicar um aumento na demanda.

## 7. Como é o I.P de partida e chegada por horário de partida do voo? Horários com mais partidas ou chegadas tem mais atrasos?

In [0]:
%sql
WITH voos AS (
	SELECT
		d.data
		, HOUR(fv.partida_prevista) as hora_partida
		,COUNT(*) AS viagens
        , SUM(IF(pontual_partida, 1, 0)) AS partidas_pontuais
        , SUM(IF(pontual_chegada, 1, 0)) AS chegadas_pontuais
        , ROUND(TRY_DIVIDE(SUM(IF(pontual_partida, 1, 0)), COUNT(*)), 3) AS IP_partida
        , ROUND(TRY_DIVIDE(SUM(IF(pontual_chegada, 1, 0)), COUNT(*)), 3) AS IP_chegada

	FROM
		mvp_engenharia_dados.gold.fato_voos fv
			JOIN mvp_engenharia_dados.gold.dim_calendario d
				ON d.sk_data = fv.sk_data_partida

	GROUP BY
        d.data
		, HOUR(fv.partida_prevista)
)
SELECT
	hora_partida 
    , avg(viagens) AS media_viagens
    , avg(partidas_pontuais) AS media_partidas_pontuais
    , avg(chegadas_pontuais) AS media_chegadas_pontuais
    , avg(IP_partida) AS media_IP_partida
    , avg(IP_chegada) AS media_IP_chegada
FROM
	voos
GROUP by hora_partida
ORDER BY hora_partida

In [0]:
%sql
WITH voos AS (
	SELECT
		d.data
		,HOUR(fv.partida_prevista) AS hora_partida
		,COUNT(*) AS viagens
		,SUM(IF(pontual_partida, 1, 0)) AS partidas_pontuais
		,SUM(IF(pontual_chegada, 1, 0)) AS chegadas_pontuais
		,ROUND(TRY_DIVIDE(SUM(IF(pontual_partida, 1, 0)), COUNT(*)), 3) AS IP_partida
		,ROUND(TRY_DIVIDE(SUM(IF(pontual_chegada, 1, 0)), COUNT(*)), 3) AS IP_chegada

	FROM
		mvp_engenharia_dados.gold.fato_voos fv
			JOIN mvp_engenharia_dados.gold.dim_calendario d
				ON d.sk_data = fv.sk_data_partida

	GROUP BY
		d.data
		,HOUR(fv.partida_prevista)
)
,pontualidade AS (
	SELECT
		hora_partida
		,AVG(viagens) AS media_viagens
		,AVG(partidas_pontuais) AS media_partidas_pontuais
		,AVG(chegadas_pontuais) AS media_chegadas_pontuais
		,AVG(IP_partida) AS media_IP_partida
		,AVG(IP_chegada) AS media_IP_chegada

	FROM
		voos

	GROUP BY
		hora_partida
)
SELECT
	CORR(media_viagens, media_ip_partida) correlacao_nviagens_pontualidade

FROM
	pontualidade

### Pontualidade por Faixa de Horário

|hora_partida|media_viagens|media_partidas_pontuais|media_chegadas_pontuais|media_IP_partida|media_IP_chegada|
|---|---|---|---|---|---|
|0|78.38|71.62|72.19|91.51%|92.25%|
|1|66.62|59.36|60.19|89.29%|90.49%|
|2|60.9|55.16|55.58|90.57%|91.22%|
|3|10.7|9.88|9.96|93.12%|93.85%|
|4|8.26|7.73|7.78|93.93%|94.52%|
|5|35.14|33.03|33.26|94.17%|94.81%|
|6|41.14|39.39|39.38|95.74%|95.72%|
|7|33.08|32.06|32.03|96.96%|96.85%|
|8|66.85|64.85|64.63|97.01%|96.71%|
|9|104.72|101.24|100.91|96.68%|96.38%|
|10|80.32|76.31|76.08|95%|94.72%|
|11|153.89|145.57|145.12|94.61%|94.32%|
|12|123.96|116.27|116.06|93.8%|93.63%|
|13|108.46|101.16|101.15|93.27%|93.29%|
|14|113.44|105.04|105.2|92.6%|92.74%|
|15|131.95|122.04|121.7|92.51%|92.28%|
|16|107.56|98.67|98.35|91.77%|91.48%|
|17|122|111.44|111.16|91.37%|91.13%|
|18|117.04|106.06|105.5|90.7%|90.23%|
|19|90.04|81.18|81.03|90.24%|90.06%|
|20|144.98|129.78|129.38|89.6%|89.34%|
|21|104.68|93.35|92.81|89.39%|88.9%|
|22|96.51|85.74|85|89.01%|88.29%|
|23|88.31|79.55|79.39|90.35%|90.2%|

> A pontualidade é melhor pela manhã até entre 5h e 11h, e declina aos poucos ao longo dia até atingir seus piores índices a noite na faixa de 20h - 22h. 

> Sobre horarios com mais viagens terem pior desempenho, não existe uma relação muito clara. O resultado da correlação foi de `-30.54%`

## 8. Qual é a média de tempo de atraso de partida e de chegada, por Companhia?

In [0]:
%sql
SELECT
	de.empresa
	, ROUND(AVG(atraso_partida_min), 1) AS media_atraso_partida
	, ROUND(AVG(atraso_chegada_min), 1) AS media_atraso_chegada

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_empresas de
			ON de.sk_empresa = fv.sk_empresa

GROUP BY
	de.empresa
ORDER BY
	media_atraso_partida
	,media_atraso_chegada

### Média de tempo de atraso por empresa (Em minutos)

|empresa|media_atraso_partida|media_atraso_chegada|
|---|---|---|
|GOL (GLO)|4.4|1.3|
|AZUL (AZU)|4.4|2.2|
|TAM (TAM)|5.2|1.9|
|SIDERAL (SID)|6.6|11.9|
|AZUL (ACN)|8.8|7.9|
|OMNI (OMI)|12.4|27.1|
|PLACAR (PLS)|15.4|21.9|
|VOEPASS (PTB)|18.8|18.6|
|ATA (ABJ)|21.7|19.3|
|MAP (PAM)|22.3|27.4|
|APUÍ (CQB)|25.3|32.5|
|AVION (AEB)|44.5|47|

## 9. Qual é a média de tempo de atraso de partida e de chegada, por Rota?

### Novamente para os casos abaixo utilizaremos um limiar tambem customizado

In [0]:
%sql
SELECT
    STDDEV(fv.atraso_partida_min) desvio_padrao
FROM
	mvp_engenharia_dados.gold.fato_voos fv


| desvio_padrao |
|---|
| 27.89 |


In [0]:
import math

z = 1.96       # 95% de confiança
sigma = 27.89  # desvio padrão do tempo de atraso, em minutos
e = 5          # margem de erro de 5 minutos

n = (z * sigma / e) ** 2

print(f"n calculado: {n:.1f}")
print(f"n mínimo: {math.ceil(n)}")

### Média de atraso por rota (Partida)

In [0]:
%sql
SELECT
	CONCAT(ori.aerodromo, ' X ', des.aerodromo) AS rota
	,COUNT(*) AS total_voos
	,AVG(atraso_partida_min) media_atraso_partida

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos ori
			ON ori.sk_aerodromo = fv.sk_aeroporto_origem
		JOIN mvp_engenharia_dados.gold.dim_aerodromos des
			ON des.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	ori.aerodromo
	,des.aerodromo

HAVING
	COUNT(*) >= 120
ORDER BY
	media_atraso_partida ASC
	,total_voos DESC
LIMIT 10

### Média de Atraso na Partida por Rota (10 melhores)

|rota|total_voos|media_atraso_partida|
|---|---|---|
|CANOAS (SBCO) X VIRACOPOS (SBKP)|431|-16.27|
|CANOAS (SBCO) X GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO (SBGR)|662|-14.78|
|REGIONAL DE SORRISO ADOLINO BEDIN (SBSO) X TANCREDO NEVES (SBCF)|193|-14.77|
|CANOAS (SBCO) X SÃO PAULO/CONGONHAS - DEPUTADO FREITAS NOBRE (SBSP)|176|-11.93|
|PRESIDENTE JUSCELINO KUBITSCHEK (SBBR) X LEITE LOPES (SBRP)|345|-10.11|
|BORBA (SWBR) X EDUARDO GOMES (SBEG)|263|-9.24|
|ORLANDO BEZERRA DE MENEZES (SBJU) X PRESIDENTE JUSCELINO KUBITSCHEK (SBBR)|200|-8.82|
|JOÃO CORRÊA DA ROCHA (SBMA) X PRESIDENTE JUSCELINO KUBITSCHEK (SBBR)|2367|-7.12|
|AEROPORTO INTERNACIONAL DO RIO GRANDE DO NORTE / SÃO GONÇALO DO AMARANTE - GOVERNADOR ALUIZIO ALVES (SBSG) X PROFESSOR ERIBELTO MANOEL REINO (SBSR)|208|-6.36|
|SENADOR PETRÔNIO PORTELLA (SBTE) X INTERNACIONAL DE BELÉM/VAL DE CANS/JÚLIO CEZAR RIBEIRO (SBBE)|439|-6.08|

In [0]:
%sql
SELECT
	CONCAT(ori.aerodromo, ' X ', des.aerodromo) AS rota
	,COUNT(*) AS total_voos
	,AVG(atraso_partida_min) media_atraso_partida

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos ori
			ON ori.sk_aerodromo = fv.sk_aeroporto_origem
		JOIN mvp_engenharia_dados.gold.dim_aerodromos des
			ON des.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	ori.aerodromo
	,des.aerodromo

HAVING
	COUNT(*) >= 120
ORDER BY
	media_atraso_partida DESC
	,total_voos DESC
LIMIT 10

### Média de Atraso na Partida por Rota (10 melhores)


|rota|total_voos|media_atraso_partida|
|---|---|---|
|LEITE LOPES (SBRP) X PRESIDENTE JUSCELINO KUBITSCHEK (SBBR)|345|60.35|
|HERCÍLIO LUZ (SBFL) X SANTA MARIA (SBSM)|132|53.25|
|ARIPUANÃ (SSOU) X JUÍNA (SWJN)|196|48.96|
|LEITE LOPES (SBRP) X SANTA GENOVEVA (SBGO)|175|44.62|
|JUÍNA (SWJN) X MARECHAL RONDON (SBCY)|211|42.2|
|URUCU (SBUY) X CARAUARI (SWCA)|292|41.09|
|GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO (SBGR) X GENERAL LEITE DE CASTRO (SWLC)|149|35.06|
|PRESIDENTE JUSCELINO KUBITSCHEK (SBBR) X ARAGUAÍNA (SWGN)|184|34.23|
|SÃO PAULO/CONGONHAS - DEPUTADO FREITAS NOBRE (SBSP) X USIMINAS (SBIP)|767|30.06|
|ARAGUAÍNA (SWGN) X PRESIDENTE JUSCELINO KUBITSCHEK (SBBR)|183|26.89|

### Média de Atraso (Chegada)

In [0]:
%sql
SELECT
	CONCAT(ori.aerodromo, ' X ', des.aerodromo) AS rota
	,COUNT(*) AS total_voos
	,AVG(atraso_chegada_min) media_atraso_chegada

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos ori
			ON ori.sk_aerodromo = fv.sk_aeroporto_origem
		JOIN mvp_engenharia_dados.gold.dim_aerodromos des
			ON des.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	ori.aerodromo
	,des.aerodromo

HAVING
	COUNT(*) >= 120
ORDER BY
	media_atraso_chegada ASC
	,total_voos DESC
LIMIT 10

### Média de Atraso na Chegada por Rota (10 Melhores)

|rota|total_voos|media_atraso_chegada|
|---|---|---|
|CANOAS (SBCO) X VIRACOPOS (SBKP)|431|-20.94|
|ORLANDO BEZERRA DE MENEZES (SBJU) X PRESIDENTE JUSCELINO KUBITSCHEK (SBBR)|200|-17.81|
|REGIONAL DE SORRISO ADOLINO BEDIN (SBSO) X TANCREDO NEVES (SBCF)|193|-15.28|
|RIO DE JANEIRO/GALEÃO - ANTÔNIO CARLOS JOBIM (SBGL) X MARECHAL RONDON (SBCY)|273|-13.8|
|MACAÉ (SBME) X VIRACOPOS (SBKP)|123|-13.68|
|PRESIDENTE PRUDENTE (SBDN) X PORTO SEGURO (SBPS)|216|-13.6|
|PROFESSOR URBANO ERNESTO STUMPF (SBSJ) X RIO DE JANEIRO/GALEÃO - ANTÔNIO CARLOS JOBIM (SBGL)|383|-13.59|
|PRESIDENTE JUSCELINO KUBITSCHEK (SBBR) X ORLANDO BEZERRA DE MENEZES (SBJU)|197|-12.76|
|BORBA (SWBR) X EDUARDO GOMES (SBEG)|263|-12.46|
|PRESIDENTE JUSCELINO KUBITSCHEK (SBBR) X LEITE LOPES (SBRP)|345|-12.3|

In [0]:
%sql
SELECT
	CONCAT(ori.aerodromo, ' X ', des.aerodromo) AS rota
	,COUNT(*) AS total_voos
	,AVG(atraso_chegada_min) media_atraso_chegada

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos ori
			ON ori.sk_aerodromo = fv.sk_aeroporto_origem
		JOIN mvp_engenharia_dados.gold.dim_aerodromos des
			ON des.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	ori.aerodromo
	,des.aerodromo

HAVING
	COUNT(*) >= 120
ORDER BY
	media_atraso_chegada DESC
	,total_voos DESC
LIMIT 10

### Média de Atraso na Chegada por Rota (10 Piores)

|rota|total_voos|media_atraso_chegada|
|---|---|---|
|LEITE LOPES (SBRP) X PRESIDENTE JUSCELINO KUBITSCHEK (SBBR)|345|58.66|
|HERCÍLIO LUZ (SBFL) X SANTA MARIA (SBSM)|132|49.47|
|ARIPUANÃ (SSOU) X JUÍNA (SWJN)|196|47.44|
|URUCU (SBUY) X CARAUARI (SWCA)|292|46.05|
|LEITE LOPES (SBRP) X SANTA GENOVEVA (SBGO)|175|45.3|
|JUÍNA (SWJN) X MARECHAL RONDON (SBCY)|211|43.34|
|GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO (SBGR) X GENERAL LEITE DE CASTRO (SWLC)|149|32.86|
|LEITE LOPES (SBRP) X SANTOS DUMONT (SBRJ)|606|30.36|
|PRESIDENTE JUSCELINO KUBITSCHEK (SBBR) X ARAGUAÍNA (SWGN)|184|29.38|
|APUÍ (SWYN) X MANICORÉ (SBMY)|200|29.23|

## 10. Qual é a média de tempo de atraso de partida e de chegada, por Aeroporto?

In [0]:
%sql
SELECT
	d.aerodromo
	,d.municipio
	,d.uf
	,COUNT(*) AS viagens
	,AVG(atraso_partida_min) media_atraso_partida

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos d
			ON d.sk_aerodromo = fv.sk_aeroporto_origem

GROUP BY
	d.aerodromo
	,d.municipio
	,d.uf

HAVING
	COUNT(*) >= 120
ORDER BY
	media_atraso_partida ASC
	,viagens DESC
LIMIT 10

### Média de Atraso de Partida por Aeroporto (10 melhores)

|aerodromo|municipio|uf|viagens|media_atraso_partida|
|---|---|---|---|---|
|CANOAS (SBCO)|CANOAS|RIO GRANDE DO SUL|1,285|-14.8|
|BORBA (SWBR)|BORBA|AMAZONAS|263|-9.24|
|AEROPORTO FÁBIO PERINI (SDLO)|CAIRU|BAHIA|126|-5.92|
|REGIONAL DE SORRISO ADOLINO BEDIN (SBSO)|SORRISO|MATO GROSSO|979|-5.38|
|PROFESSOR URBANO ERNESTO STUMPF (SBSJ)|SÃO JOSÉ DOS CAMPOS|SÃO PAULO|397|-4.84|
|GARANHUNS (SNGN)|GARANHUNS|PERNAMBUCO|421|-4.6|
|MONTE ALEGRE (SNMA)|MONTE ALEGRE|PARÁ|164|-4.57|
|JOÃO CORRÊA DA ROCHA (SBMA)|MARABÁ|PARÁ|5,642|-4.19|
|AERÓDROMO PÚBLICO DE BONITO (SBDB)|BONITO|MATO GROSSO DO SUL|1,224|-3.71|
|ORIXIMINÁ (SNOX)|ORIXIMINÁ|PARÁ|163|-3.61|

In [0]:
%sql
SELECT
	d.aerodromo
	,d.municipio
	,d.uf
	,COUNT(*) AS viagens
	,AVG(atraso_partida_min) media_atraso_partida

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos d
			ON d.sk_aerodromo = fv.sk_aeroporto_origem

GROUP BY
	d.aerodromo
	,d.municipio
	,d.uf

HAVING
	COUNT(*) >= 120
ORDER BY
	media_atraso_partida DESC
	,viagens DESC
LIMIT 10

### Média de atraso de Partida por Aeroporto (10 piores)

|aerodromo|municipio|uf|viagens|media_atraso_partida|
|---|---|---|---|---|
|JUÍNA (SWJN)|JUÍNA|MATO GROSSO|211|42.2|
|CARAUARI (SWCA)|CARAUARI|AMAZONAS|399|32.13|
|LORENZO (SNCL)|CAIRU|BAHIA|1,434|24.89|
|URUCU (SBUY)|COARI|AMAZONAS|2,035|24.69|
|ARIPUANÃ (SSOU)|ARIPUANÃ|MATO GROSSO|684|20.8|
|MUNICIPAL DE LINHARES (SNLN)|LINHARES|ESPÍRITO SANTO|470|19.73|
|BARRA DO GARÇAS (SBBW)|BARRA DO GARÇAS|MATO GROSSO|653|18.78|
|MARICÁ (SBMI)|MARICÁ|RIO DE JANEIRO|248|18.14|
|AEROPORTO REGIONAL DE CANOA QUEBRADA DRAGÃO DO MAR (SBAC)|ARACATI|CEARÁ|162|17.96|
|ARARIPINA (SNAB)|ARARIPINA|PERNAMBUCO|407|16.26|

In [0]:
%sql
SELECT
	d.aerodromo
	,d.municipio
	,d.uf
	,COUNT(*) AS viagens
	,AVG(atraso_chegada_min) media_atraso_chegada

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos d
			ON d.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	d.aerodromo
	,d.municipio
	,d.uf

HAVING
	COUNT(*) >= 120
ORDER BY
	media_atraso_chegada ASC
	,viagens DESC
LIMIT 10

### Média de Atraso de Chegada por Aeroporto (10 Melhores)

|aerodromo|municipio|uf|viagens|media_atraso_chegada|
|---|---|---|---|---|
|BORBA (SWBR)|BORBA|AMAZONAS|262|-8.58|
|GARANHUNS (SNGN)|GARANHUNS|PERNAMBUCO|399|-6.23|
|TROMBETAS (SBTB)|ORIXIMINÁ|PARÁ|615|-4.96|
|PROFESSOR URBANO ERNESTO STUMPF (SBSJ)|SÃO JOSÉ DOS CAMPOS|SÃO PAULO|383|-3.36|
|CRUZEIRO DO SUL (SBCZ)|CRUZEIRO DO SUL|ACRE|704|-2.35|
|AEROPORTO REGIONAL DE GUARAPUAVA - TANCREDO THOMÁS DE FARIA (SSGG)|GUARAPUAVA|PARANÁ|565|-2.34|
|JOÃO CORRÊA DA ROCHA (SBMA)|MARABÁ|PARÁ|5562|-1.92|
|MAESTRO WILSON FONSECA (SBSN)|SANTARÉM|PARÁ|7441|-1|
|SÃO GABRIEL DA CACHOEIRA (SBUA)|SÃO GABRIEL DA CACHOEIRA|AMAZONAS|404|-0.9|
|JOÃO DURVAL CARNEIRO (SDIY)|FEIRA DE SANTANA|BAHIA|352|-0.89|

In [0]:
%sql
SELECT
	d.aerodromo
	,d.municipio
	,d.uf
	,COUNT(*) AS viagens
	,AVG(atraso_chegada_min) media_atraso_chegada

FROM
	mvp_engenharia_dados.gold.fato_voos fv
		JOIN mvp_engenharia_dados.gold.dim_aerodromos d
			ON d.sk_aerodromo = fv.sk_aeroporto_destino

GROUP BY
	d.aerodromo
	,d.municipio
	,d.uf

HAVING
	COUNT(*) >= 120
ORDER BY
	media_atraso_chegada DESC
	,viagens DESC
LIMIT 10

### Média de Atraso de Chegada por Aeroporto (10 Piores)

|aerodromo|municipio|uf|viagens|media_atraso_chegada|
|---|---|---|---|---|
|CARAUARI (SWCA)|CARAUARI|AMAZONAS|400|50.24|
|JUÍNA (SWJN)|JUÍNA|MATO GROSSO|196|47.44|
|APUÍ (SWYN)|APUÍ|AMAZONAS|276|28.49|
|LORENZO (SNCL)|CAIRU|BAHIA|1565|20.59|
|MUNICIPAL DE LINHARES (SNLN)|LINHARES|ESPÍRITO SANTO|474|20.45|
|URUCU (SBUY)|COARI|AMAZONAS|2040|20.43|
|ARIPUANÃ (SSOU)|ARIPUANÃ|MATO GROSSO|714|19.93|
|AEROPORTO REGIONAL DE CANOA QUEBRADA DRAGÃO DO MAR (SBAC)|ARACATI|CEARÁ|163|19.1|
|ARAGUAÍNA (SWGN)|ARAGUAÍNA|TOCANTINS|292|18.96|
|BARRA DO GARÇAS (SBBW)|BARRA DO GARÇAS|MATO GROSSO|660|17.56|

## Resumo da Etapa de Perguntas de Negócio

Este notebook respondeu 10 perguntas de negócio sobre pontualidade de voos domésticos no Brasil (Jan/2023 a Jul/2026), utilizando o star schema da camada gold (`fato_voos` + `dim_empresas` + `dim_aerodromos` + `dim_calendario`) com 2.732.331 registros.

### Metodologia

Para rankings por rota e aeroporto, aplicamos intervalos de confiança para garantir amostras representativas:

| Tipo de análise | Fórmula | Parâmetros | N mínimo |
| --- | --- | --- | --- |
| Proporção (I.P.) | `n = z²·p·(1-p) / e²` | z=1,96 · p=0,922 · e=0,03 | **307** |
| Média (atraso) | `n = (z·σ / e)²` | z=1,96 · σ=27,89 · e=5 | **120** |

### Principais Achados

**Por companhia (Q1-Q2):**
* I.P. geral: **92,2%** na partida e **92,1%** na chegada
* TAM (92,6%), GOL (92,6%) e AZUL (92,3%) ficam acima da média; operadoras menores como AVION e MAP têm os piores índices
* Maioria das empresas melhora de 2023 para 2025; 2026 (parcial) indica estabilização

**Por rota (Q3):**
* Melhor: CANOAS (SBCO) X VIRACOPOS (SBKP) com **99,5%** de pontualidade
* Pior: RIBEIRÃO PRETO (SBRP) X BRASÍLIA (SBBR) com **67,2%**. Ribeirão Preto é um gargalo recorrente, aparecendo também nas piores por atraso médio (60 min)

**Por aeroporto (Q4):**
* Melhor partida: CANOAS (SBCO) com **99%**
* Pior partida: CARAUARI (SWCA) com **67,4%**. Aeroportos da Amazônia (Carauari, Urucu, Coari) concentram os piores índices

**Sazonalidade mensal (Q5):**
* Melhor mês: **Maio** (94,5%)
* Pior mês: **Agosto** (82,3%), com queda abrupta de ~10 p.p. em relação aos meses vizinhos
* Novembro e dezembro também apresentam declínio (87-90%)

**Por dia da semana (Q6):**
* Melhores: **Domingo** (93,5%) e **Sábado** (93,1%)
* Piores: **Quinta** (91,1%) e **Sexta** (90,7%), com queda de ~3 p.p. no fim da semana útil

**Por horário (Q7):**
* Melhor faixa: **7h-9h** (96-97%)
* Pior faixa: **20h-22h** (89%)
* Correlação volume x pontualidade: **-30,5%** (negativa fraca): mais voos tende a piorar, mas não é determinante

**Atraso médio (Q8-Q10):**
* Menor atraso: GOL (4,4 min partida / 1,3 min chegada) e TAM (5,2 / 1,9)
* Maior atraso: AVION (44,5 / 47 min)
* Rota com maior atraso: RIBEIRÃO PRETO X BRASÍLIA (**60 min** partida, 59 min chegada)
* Aeroporto com maior atraso: JUÍNA-MT (**42 min** partida) e CARAUARI-AM (**50 min** chegada)

### Conclusão

Três padrões se destacam:
1. **Geografia:** aeroportos da Amazônia (Carauari, Urucu, Coari, Juína) e Ribeirão Preto concentram os piores indicadores
2. **Tempo:** pontualidade cai ao longo do dia (manhã > noite), na quinta/sexta-feira, e em agosto/novembro-dezembro
3. **Operador:** as três grandes (TAM, GOL, AZUL) mantêm índices acima da média, enquanto operadoras regionais têm maior variabilidade

As 10 perguntas foram respondidas com amostras representativas quando necessário.